# 📚 Day 01: Stack Fundamentals — Last In, First Out

---

## Think of a Stack of Plates

Literally — a stack of plates in a cafeteria. You can only add a plate on top, and you can only take a plate from the top. You can't grab one from the middle without toppling everything.

That's **LIFO — Last In, First Out.** The last thing you pushed is the first thing you pop.

```
╔══════════════════════════════════════════╗
║  STACK = LIFO (Last In, First Out)       ║
╠══════════════════════════════════════════╣
║                                          ║
║  push(x)  → add x to top      O(1)      ║
║  pop()    → remove & return top O(1)     ║
║  peek()   → look at top        O(1)      ║
║  isEmpty  → is stack empty?     O(1)     ║
║                                          ║
║  In Python: just use a list!             ║
║    push = append()                        ║
║    pop  = pop()                           ║
║    peek = stack[-1]                       ║
║                                          ║
╚══════════════════════════════════════════╝
```

**Where do stacks show up in real life?**
- Browser back button (stack of URLs)
- Undo in any editor (stack of operations)
- Function call stack (recursion!)
- Matching parentheses, brackets, tags

---

In [ ]:
# Stack in Python — it's just a list
stack = []

# Push
stack.append(1)
stack.append(2)
stack.append(3)
print(f"Stack: {stack}")        # [1, 2, 3]  (3 is on top)

# Peek
print(f"Top: {stack[-1]}")      # 3

# Pop
print(f"Pop: {stack.pop()}")    # 3
print(f"Pop: {stack.pop()}")    # 2
print(f"Stack: {stack}")        # [1]

# Empty check
print(f"Empty? {len(stack) == 0}")  # False
stack.pop()
print(f"Empty? {len(stack) == 0}")  # True

---

## 🎯 Problem 1: Valid Parentheses (LC #20)

The **classic** stack problem. Shows up in practically every interview prep guide.

```
Input: "({[]})"  →  True
Input: "([)]"    →  False
Input: "{[}"     →  False
```

**Why is this a stack problem?** When you see an opening bracket, you expect its closing bracket later — but any brackets opened AFTER it must close FIRST. That's LIFO!

```
Processing "({[]})":
  (  → push '('
  {  → push '{'
  [  → push '['
  ]  → pop '[' — matches! ✓
  }  → pop '{' — matches! ✓
  )  → pop '(' — matches! ✓
  Stack empty → Valid!
```

In [ ]:
def isValid(s):
    """
    Check if brackets are valid using a stack.
    Time: O(n), Space: O(n)
    """
    stack = []
    matching = {')': '(', '}': '{', ']': '['}
    
    for ch in s:
        if ch in matching:  # It's a closing bracket
            # Check if top of stack has the matching opener
            if not stack or stack[-1] != matching[ch]:
                return False
            stack.pop()
        else:  # It's an opening bracket
            stack.append(ch)
    
    return len(stack) == 0  # Valid only if all openers were matched

print(isValid("({[]})"))  # True
print(isValid("([)]"))    # False
print(isValid("{[}"))     # False
print(isValid("()"))      # True
print(isValid("("))       # False (unmatched opener)

---

## 🎯 Problem 2: Min Stack (LC #155)

Design a stack that supports push, pop, top, AND `getMin` — all in O(1).

```
MinStack();
push(-2);  push(0);  push(-3);
getMin();  → -3
pop();     → removes -3
getMin();  → -2  (still knows the min!)
```

**The trick:** Maintain a second stack (or store tuples) that tracks the minimum at each level. When you push, also push the current minimum. When you pop, both stacks pop together.

```
Main stack:   [-2]  [-2, 0]  [-2, 0, -3]
Min stack:    [-2]  [-2, -2] [-2, -2, -3]
                                     ↑ min is always at top of min stack
```

In [ ]:
class MinStack:
    """
    Stack with O(1) getMin using a parallel min-tracking stack.
    """
    def __init__(self):
        self.stack = []      # Main stack
        self.min_stack = []  # Tracks minimum at each level
    
    def push(self, val):
        self.stack.append(val)
        # Push the new minimum (min of val and current min)
        current_min = min(val, self.min_stack[-1] if self.min_stack else val)
        self.min_stack.append(current_min)
    
    def pop(self):
        self.stack.pop()
        self.min_stack.pop()
    
    def top(self):
        return self.stack[-1]
    
    def getMin(self):
        return self.min_stack[-1]

ms = MinStack()
ms.push(-2)
ms.push(0)
ms.push(-3)
print(f"Min: {ms.getMin()}")  # -3
ms.pop()
print(f"Top: {ms.top()}")    # 0
print(f"Min: {ms.getMin()}")  # -2 (still knows!)

---

## 🎯 Problem 3: Decode String (LC #394)

This one is sneaky — it looks like a string problem but it's fundamentally a stack problem because of the **nested** structure.

```
Input:  "3[a2[c]]"  →  Output: "accaccacc"
Input:  "2[abc]3[cd]ef"  →  Output: "abcabccdcdcdef"
```

**Why stack?** The brackets can be **nested**. When you hit a `[`, you need to "save" your current work and start fresh. When you hit `]`, you "restore" and combine. That's push/pop.

**Algorithm:**
- See a digit → build the number (could be multi-digit like `12[a]`)
- See `[` → push (current_string, current_number) onto stack, reset both
- See `]` → pop (prev_string, repeat_count), combine: prev_string + current_string × repeat_count
- See a letter → add to current_string

In [ ]:
def decodeString(s):
    """
    Decode nested encoded string using a stack.
    Time: O(n * max_k), Space: O(n)
    """
    stack = []  # Will hold (previous_string, repeat_count) tuples
    current_str = ""
    current_num = 0
    
    for ch in s:
        if ch.isdigit():
            current_num = current_num * 10 + int(ch)  # Handle multi-digit
        elif ch == '[':
            # Save current state and reset
            stack.append((current_str, current_num))
            current_str = ""
            current_num = 0
        elif ch == ']':
            # Pop and combine
            prev_str, repeat = stack.pop()
            current_str = prev_str + current_str * repeat
        else:
            current_str += ch
    
    return current_str

print(decodeString("3[a2[c]]"))      # accaccacc
print(decodeString("2[abc]3[cd]ef")) # abcabccdcdcdef
print(decodeString("10[a]"))         # aaaaaaaaaa

---

## 💡 When to Think "Stack"?

```
╔══════════════════════════════════════════════════════════════════════╗
║  STACK SIGNAL WORDS                                                  ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  • "matching" / "valid" brackets/parentheses/tags                   ║
║  • "nested" structures (decode, flatten)                            ║
║  • "undo" / "back" / "most recent first"                           ║
║  • "next greater" / "next smaller" (→ monotonic stack)              ║
║  • Expression evaluation / calculators                               ║
║  • DFS (depth-first search) — stack replaces recursion              ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Pattern |
|---|---------|------------|--------|
| 20 | Valid Parentheses | Easy | Matching |
| 155 | Min Stack | Medium | Parallel tracking |
| 394 | Decode String | Medium | Nested processing |
| 682 | Baseball Game | Easy | Simulation |
| 225 | Implement Stack using Queues | Easy | Design |
| 150 | Evaluate Reverse Polish Notation | Medium | Expression eval |
| 71 | Simplify Path | Medium | String + stack |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Stack = LIFO. In Python, it's just a list with append/pop.     │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Matching brackets: push openers, pop & match on closers        │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Min Stack: parallel min stack tracks minimum at each depth     │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Nested structures: push state on '[', restore on ']'           │
├──────────────────────────────────────────────────────────────────────┤
│  📌 See "matching", "nested", or "most recent"? Think stack.       │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Monotonic Stack

The elegant O(n) trick for "next greater element" problems.

---

### 🎉 Stack Fundamentals Set!
LIFO is everywhere once you learn to see it.